In [1]:
%matplotlib notebook

In [2]:
from pathlib import Path
from glob import glob
import pandas as pd
import re
import os

In [3]:
from bikipy.behaviour.y_maze import reduce_str_sequence, spontaneous_alterntations
from bikipy.border.parallelogram.classes import ParallelogramBorder
from bikipy.border.triangular import TriangularBorder
from bikipy.readers import DeepLabCutReader

In [4]:
WORKING_DIR = Path("C:/Users/Can/Projects/Neuroscience/bikipy/examples/data/")
DATA_DIR = Path("C:/Users/Can/Projects/Neuroscience/Imen/data/y_maze")

In [5]:
BORDER_IMG_PATH = WORKING_DIR / "images" / "y_maze" / "phd.png"
assert BORDER_IMG_PATH.exists(), f"The image file doesn't exist in {BORDER_IMG_PATH}"
border_img_path_str = str(BORDER_IMG_PATH)

borders = [
        ParallelogramBorder(
            base=[[308.81687801, 193.11825], [288.30224458, 234.14751686]],
            apex=[[174.33205886, 120.17733115], [151.53802172, 158.92719429]],
            guiding_image=border_img_path_str, label="A"
        ),
        ParallelogramBorder(
            base=[[309.95657987, 193.11825], [335.03002072, 234.14751686]],
            apex=[[441.02229344, 116.75822557], [461.53692687, 154.36838686]],
            guiding_image=border_img_path_str, label="B"
        ),
        ParallelogramBorder(
            base=[[290.58164829, 234.14751686], [335.03002072, 234.14751686]],
            apex=[[294.00075387, 394.84547872], [338.44912629, 394.84547872]],
            guiding_image=border_img_path_str, label="C"
        )
    ]

inferior = [
    TriangularBorder(
            base_a=(289.62987012987014, 235.08441558441552),
            base_b=(332.48701298701303, 235.08441558441552),
            apex=(307.8116883116883, 198.72077922077915),
            label="X"
        )
]

In [6]:
exp_id_finder = re.compile("\d+")
data_dict = {}
for subdir in os.listdir(str(DATA_DIR)):
    print(subdir)
    for file_path in glob(os.path.join(str(DATA_DIR / subdir), "**.h5")):
        exp_id = exp_id_finder.findall(Path(file_path).stem)[0]
        print(exp_id)
        dlc_obj = DeepLabCutReader.from_hdf(
            file_path, (640, 480), midpoint_groups=(("left_ear", "right_ear"),)
        )
        data_dict[(subdir, exp_id)] = ParallelogramBorder.detect_sequential_border_presence(
            dlc_obj["mid-left_ear-right_ear"], borders, overlap_inferior=inferior
        )

In [ ]:
reduced = {}
for info, location_per_frame in data_dict.items():
    reduced[info] = reduce_str_sequence(location_per_frame)

In [ ]:
exp_info_vs_spontaneous_alterntations = {}
for info, arm_location_sequence in reduced.items():
    exp_info_vs_spontaneous_alterntations[info] = spontaneous_alterntations(
        arm_location_sequence, exclude="X"
    )

In [ ]:
# pd.DataFrame.from_dict(
#     exp_info_vs_spontaneous_alterntations, "index", columns=["Spontaneous Alterntations"]
# ).to_excel("Spontaneous Alterntations.xlsx")